# CreditSense — Final Pipeline (Team Slavs)

End-to-end, Colab-runnable notebook that reproduces our Kaggle submission for the AI1215 CreditSense challenge.

**Team:** Slavs &mdash; Makar Ulesov, Ivan Kanev, Delyan Hristov.  
**Approach:** LightGBM + XGBoost + CatBoost base learners (+ ordinal-regression trick for Task A) with cross-task stacking and convex ensemble weights optimised on OOF.  

### How to run
1. Upload `credit_train.csv` and `credit_test.csv` into the same directory as this notebook.
2. `Runtime → Run all` (Colab) or `Kernel → Restart & Run All` (Jupyter).
3. The submission is written to `submission.csv` after the last cell.

## 0. Install dependencies (Colab only)

In [ ]:
# In Colab, uncomment the next line. Locally, pip-install via requirements.txt.
# !pip install -q lightgbm xgboost catboost optuna

## 1. Reproducibility + I/O

In [ ]:
import os, random, json, time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, r2_score
from sklearn.model_selection import StratifiedKFold
from scipy.optimize import minimize

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRegressor

SEED = 42
N_FOLDS = 5
N_CLASSES = 5
RATE_MIN, RATE_MAX = 4.99, 35.99

random.seed(SEED); np.random.seed(SEED); os.environ['PYTHONHASHSEED'] = str(SEED)

# Colab default CWD is /content. Adjust if you mount Drive elsewhere.
DATA_DIR = Path('.')
train = pd.read_csv(DATA_DIR/'credit_train.csv')
test = pd.read_csv(DATA_DIR/'credit_test.csv')
if 'Id' not in train.columns: train.insert(0, 'Id', np.arange(len(train)))
if 'Id' not in test.columns:  test.insert(0, 'Id', np.arange(len(test)))
print('train:', train.shape, '  test:', test.shape)

## 2. Feature engineering

Each block carries financial intuition. See Section 2 of the written report for the ablation table.

In [ ]:
def _has(df, *cols): return all(c in df.columns for c in cols)
def _safe_div(a, b):
    out = a.astype(float) / b.replace(0, np.nan).astype(float)
    return out.replace([np.inf, -np.inf], np.nan).fillna(0.0)

def engineer_features(df):
    out = df.copy()

    # ---- Balance sheet ----
    liab_cols = [c for c in ['MortgageOutstandingBalance','StudentLoanOutstandingBalance',
                             'AutoLoanOutstandingBalance'] if c in out.columns]
    if liab_cols:
        out['feat_TotalLiabilities'] = out[liab_cols].fillna(0).sum(axis=1)
    if _has(out,'TotalAssets') and 'feat_TotalLiabilities' in out.columns:
        out['feat_NetWorth'] = out['TotalAssets'] - out['feat_TotalLiabilities']
        out['feat_DebtToAssets'] = _safe_div(out['feat_TotalLiabilities'], out['TotalAssets'])
    if _has(out,'SavingsBalance','CheckingBalance'):
        out['feat_LiquidCash'] = out['SavingsBalance'].fillna(0) + out['CheckingBalance'].fillna(0)
    if _has(out,'feat_LiquidCash','RequestedLoanAmount'):
        out['feat_LiquidToLoan'] = _safe_div(out['feat_LiquidCash'], out['RequestedLoanAmount'])
    if _has(out,'feat_LiquidCash','TotalMonthlyIncome'):
        out['feat_CashRunwayMonths'] = _safe_div(out['feat_LiquidCash'], out['TotalMonthlyIncome'])
    if _has(out,'MortgageOutstandingBalance','PropertyValue'):
        out['feat_MortgageLTV'] = _safe_div(out['MortgageOutstandingBalance'], out['PropertyValue'])
    if _has(out,'InvestmentPortfolioValue','TotalAssets'):
        out['feat_InvestShare'] = _safe_div(out['InvestmentPortfolioValue'], out['TotalAssets'])
    if _has(out,'SavingsBalance','RequestedLoanAmount'):
        out['feat_PostLoanSavings'] = out['SavingsBalance'].fillna(0) - out['RequestedLoanAmount']

    # ---- Credit behaviour (delinquency + derogatory + inquiries) ----
    late30 = out.get('NumberOfLatePayments30Days', pd.Series(0, index=out.index))
    late60 = out.get('NumberOfLatePayments60Days', pd.Series(0, index=out.index))
    late90 = out.get('NumberOfLatePayments90Days', pd.Series(0, index=out.index))
    out['feat_DelinquencyScore'] = 1*late30.fillna(0) + 3*late60.fillna(0) + 9*late90.fillna(0)
    out['feat_AnyDelinquency']   = ((late30.fillna(0)+late60.fillna(0)+late90.fillna(0))>0).astype(np.int8)
    out['feat_SevereDelinquency']= (late90.fillna(0)>0).astype(np.int8)
    deroc = {'NumberOfChargeOffs':5,'NumberOfCollections':3,
             'NumberOfPublicRecords':4,'NumberOfBankruptcies':10}
    score = pd.Series(0.0, index=out.index); any_d = pd.Series(0, index=out.index)
    for c,w in deroc.items():
        if c in out.columns:
            v = out[c].fillna(0); score += w*v; any_d += (v>0).astype(int)
    out['feat_DerogatoryScore'] = score
    out['feat_AnyDerogatory']   = (any_d>0).astype(np.int8)
    out['feat_BadEventsTotal']  = out['feat_DelinquencyScore'] + out['feat_DerogatoryScore']
    hi12 = out.get('NumberOfHardInquiries12Mo', pd.Series(0, index=out.index)).fillna(0)
    hi24 = out.get('NumberOfHardInquiries24Mo', pd.Series(0, index=out.index)).fillna(0)
    out['feat_RecentInquiryShare']  = _safe_div(hi12, hi24)
    out['feat_HighInquiryActivity'] = (hi12>3).astype(np.int8)
    if 'RevolvingUtilizationRate' in out.columns:
        util = out['RevolvingUtilizationRate'].fillna(0)
        out['feat_HighUtilization'] = (util>0.9).astype(np.int8)
        out['feat_MaxedOut']        = (util>=1.0).astype(np.int8)
        out['feat_UtilBucket']      = pd.cut(util, bins=[-.01,.1,.3,.6,.9,np.inf],
                                              labels=False).astype(np.int8)
    if _has(out,'NumberOfSatisfactoryAccounts','NumberOfOpenAccounts'):
        out['feat_SatisfactoryRatio'] = _safe_div(out['NumberOfSatisfactoryAccounts'],
                                                   out['NumberOfOpenAccounts'])
    if _has(out,'TotalCreditLimit','RevolvingUtilizationRate'):
        out['feat_CreditUsed'] = out['TotalCreditLimit'].fillna(0)*out['RevolvingUtilizationRate'].fillna(0)
    if _has(out,'NumberOfOpenAccounts','NumberOfCreditCards'):
        out['feat_InstallmentAccounts'] = out['NumberOfOpenAccounts'].fillna(0) - out['NumberOfCreditCards'].fillna(0)

    # ---- History maturity ----
    if 'CreditHistoryLengthMonths' in out.columns:
        chl = out['CreditHistoryLengthMonths'].fillna(0)
        out['feat_ThinFile']  = (chl<24).astype(np.int8)
        out['feat_ThickFile'] = (chl>120).astype(np.int8)
    if _has(out,'OldestAccountAgeMonths','AverageAccountAgeMonths'):
        out['feat_NewAccountSpree'] = out['OldestAccountAgeMonths'].fillna(0) - out['AverageAccountAgeMonths'].fillna(0)
    if _has(out,'Age','CreditHistoryLengthMonths'):
        out['feat_CreditAgeRatio'] = _safe_div(out['CreditHistoryLengthMonths'],
                                                (out['Age'].fillna(25)-18).clip(lower=1)*12)

    # ---- Demographics / stability ----
    if 'Age' in out.columns:
        out['feat_AgeBucket'] = pd.cut(out['Age'].fillna(out['Age'].median()),
                                       bins=[-1,25,35,45,55,65,200], labels=False).astype(np.int8)
    if _has(out,'EmploymentLengthYears','Age'):
        out['feat_EmpStability'] = _safe_div(out['EmploymentLengthYears'].fillna(0),
                                              (out['Age']-18).clip(lower=1))
    if _has(out,'AnnualIncome','EmploymentLengthYears'):
        out['feat_IncomePerEmpYear'] = _safe_div(out['AnnualIncome'],
                                                  out['EmploymentLengthYears'].fillna(0)+1)
    if _has(out,'YearsAtCurrentEmployer','EmploymentLengthYears'):
        out['feat_SameEmployerShare'] = _safe_div(out['YearsAtCurrentEmployer'].fillna(0),
                                                   out['EmploymentLengthYears'].fillna(0)+0.5).clip(0,1)
    if 'ResidencyYears' in out.columns:
        out['feat_StableResident'] = (out['ResidencyYears'].fillna(0)>3).astype(np.int8)
    if 'HomeOwnership' in out.columns:
        out['feat_IsHomeOwner'] = out['HomeOwnership'].astype(str).str.upper().str.contains(
            'OWN|MORTGAGE', regex=True, na=False).astype(np.int8)
    if 'NumberOfDependents' in out.columns:
        out['feat_HasDependents'] = (out['NumberOfDependents'].fillna(0)>0).astype(np.int8)
    if _has(out,'NumberOfDependentsUnder18','NumberOfDependents'):
        out['feat_KidsShare'] = _safe_div(out['NumberOfDependentsUnder18'], out['NumberOfDependents'])
    if 'IncomeVerified' in out.columns:
        out['feat_IncomeVerified'] = out['IncomeVerified'].fillna(0).astype(np.int8)
    if _has(out,'SecondaryMonthlyIncome','MonthlyGrossIncome'):
        out['feat_HasSecondaryIncome'] = (out['SecondaryMonthlyIncome'].fillna(0)>0).astype(np.int8)
        out['feat_SecondaryIncomeShare'] = _safe_div(out['SecondaryMonthlyIncome'].fillna(0),
                                                      out['MonthlyGrossIncome']+out['SecondaryMonthlyIncome'].fillna(0))

    # ---- Loan request ----
    if _has(out,'RequestedLoanAmount','RequestedTermMonths'):
        out['feat_MonthlyPrincipal'] = _safe_div(out['RequestedLoanAmount'], out['RequestedTermMonths'])
    if 'RequestedTermMonths' in out.columns:
        out['feat_LongTerm']  = (out['RequestedTermMonths']>=60).astype(np.int8)
        out['feat_ShortTerm'] = (out['RequestedTermMonths']<=24).astype(np.int8)
    if _has(out,'RequestedLoanAmount','TotalAssets'):
        out['feat_LoanToAssets'] = _safe_div(out['RequestedLoanAmount'], out['TotalAssets'])
    if _has(out,'MonthlyPaymentEstimate','feat_LiquidCash'):
        out['feat_PaymentReserveMonths'] = _safe_div(out['feat_LiquidCash'], out['MonthlyPaymentEstimate'])
    if 'CollateralType' in out.columns:
        none_like = out['CollateralType'].astype(str).str.upper().str.strip().isin(['NONE','NAN','NA',''])
        out['feat_HasCollateral'] = (~none_like).astype(np.int8)
    if 'HasCoApplicant' in out.columns: out['feat_HasCoApplicant'] = out['HasCoApplicant'].fillna(0).astype(np.int8)
    if 'PreviousLoanWithBank' in out.columns: out['feat_RepeatCustomer'] = out['PreviousLoanWithBank'].fillna(0).astype(np.int8)

    # ---- Log-transforms ----
    money_like = [c for c in out.columns if not c.startswith('feat_log_')
                  and any(k in c.lower() for k in ('income','amount','balance','value','asset'))
                  and pd.api.types.is_numeric_dtype(out[c])]
    for c in money_like: out[f'feat_log_{c}'] = np.log1p(out[c].fillna(0).clip(lower=0))

    # ---- Interactions ----
    if _has(out,'DebtToIncomeRatio','RevolvingUtilizationRate'):
        out['feat_DTI_x_Util'] = out['DebtToIncomeRatio'].fillna(0)*out['RevolvingUtilizationRate'].fillna(0)
    if _has(out,'LoanToIncomeRatio','RequestedTermMonths'):
        out['feat_LTI_x_Term'] = out['LoanToIncomeRatio'].fillna(0)*out['RequestedTermMonths'].fillna(0)
    if _has(out,'Age','feat_DelinquencyScore'):
        out['feat_DelinquencyPerYear'] = _safe_div(out['feat_DelinquencyScore'], (out['Age']-18).clip(lower=1))
    if _has(out,'PaymentToIncomeRatio','feat_BadEventsTotal'):
        out['feat_PTI_x_BadEvents'] = out['PaymentToIncomeRatio'].fillna(0)*out['feat_BadEventsTotal']
    if _has(out,'CreditHistoryLengthMonths','feat_DerogatoryScore'):
        out['feat_DerogPerHistoryYear'] = _safe_div(out['feat_DerogatoryScore'],
                                                     out['CreditHistoryLengthMonths'].fillna(0)/12+1)

    out = out.replace([np.inf,-np.inf], np.nan)
    num_cols = out.select_dtypes(include=np.number).columns
    out[num_cols] = out[num_cols].fillna(0.0)
    return out

train_fe = engineer_features(train)
test_fe  = engineer_features(test)
print('after FE:', train_fe.shape, test_fe.shape)

## 3. Preprocessing: missing indicators, imputation, encoding

In [ ]:
TARGET_A, TARGET_B, ID_COL = 'RiskTier', 'InterestRate', 'Id'
STRUCT_ZERO = ['PropertyValue','MortgageOutstandingBalance','StudentLoanOutstandingBalance',
               'AutoLoanOutstandingBalance','InvestmentPortfolioValue','VehicleValue',
               'CollateralValue','SecondaryMonthlyIncome']

y_tier = train_fe[TARGET_A].astype(int).copy()
y_rate = train_fe[TARGET_B].astype(float).copy()
ids_test = test_fe[ID_COL].copy()

tr = train_fe.drop(columns=[TARGET_A, TARGET_B, ID_COL])
te = test_fe.drop(columns=[ID_COL])
common = [c for c in tr.columns if c in te.columns]
tr, te = tr[common], te[common]

# Missing indicators BEFORE imputation
for c in tr.columns:
    if tr[c].isna().any() or te[c].isna().any():
        tr[f'{c}_was_missing'] = tr[c].isna().astype(np.int8)
        te[f'{c}_was_missing'] = te[c].isna().astype(np.int8)

# Numeric imputation: structural-zero candidates → 0, others → median.
num_cols = [c for c in tr.columns if pd.api.types.is_numeric_dtype(tr[c])]
for c in num_cols:
    fill = 0.0 if c in STRUCT_ZERO else tr[c].median()
    tr[c] = tr[c].fillna(fill); te[c] = te[c].fillna(fill)

# Winsorise heavy right tails on train, apply to test.
for c in num_cols:
    if any(k in c.lower() for k in ('income','amount','balance','value','loan')):
        cap = tr[c].quantile(0.995)
        tr[c] = np.minimum(tr[c], cap); te[c] = np.minimum(te[c], cap)

# Categorical encoding: low-card one-hot, high-card K-fold target encode + integer codes.
cat_cols = [c for c in tr.columns if not pd.api.types.is_numeric_dtype(tr[c])]
for c in cat_cols:
    tr[c] = tr[c].fillna('NA').astype(str); te[c] = te[c].fillna('NA').astype(str)

low_card  = [c for c in cat_cols if tr[c].nunique() <= 8]
high_card = [c for c in cat_cols if c not in low_card]

oh_tr = pd.get_dummies(tr[low_card], prefix=low_card, dtype=np.int8) if low_card else pd.DataFrame(index=tr.index)
oh_te = pd.get_dummies(te[low_card], prefix=low_card, dtype=np.int8) if low_card else pd.DataFrame(index=te.index)
oh_tr, oh_te = oh_tr.align(oh_te, join='outer', axis=1, fill_value=0)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
te_tr = pd.DataFrame(index=tr.index); te_te = pd.DataFrame(index=te.index)
if high_card:
    gmean = float(y_rate.mean())
    for c in high_card:
        enc_tr = pd.Series(gmean, index=tr.index, dtype=float)
        for tri, vai in skf.split(tr, y_tier):
            agg = y_rate.iloc[tri].groupby(tr[c].iloc[tri]).agg(['sum','count'])
            sm = (agg['sum'] + 20*gmean)/(agg['count']+20)
            enc_tr.iloc[vai] = tr[c].iloc[vai].map(sm).fillna(gmean).to_numpy()
        te_tr[f'{c}_te'] = enc_tr
        agg_all = y_rate.groupby(tr[c]).agg(['sum','count'])
        sm_all = (agg_all['sum'] + 20*gmean)/(agg_all['count']+20)
        te_te[f'{c}_te'] = te[c].map(sm_all).fillna(gmean).astype(float)

code_tr = pd.DataFrame(index=tr.index); code_te = pd.DataFrame(index=te.index)
for c in high_card:
    combined = pd.concat([tr[c], te[c]], axis=0)
    codes, _ = pd.factorize(combined, sort=True)
    code_tr[f'{c}_code'] = codes[:len(tr)].astype(np.int32)
    code_te[f'{c}_code'] = codes[len(tr):].astype(np.int32)

X_train = pd.concat([tr.drop(columns=cat_cols), oh_tr, te_tr, code_tr], axis=1)
X_test  = pd.concat([te.drop(columns=cat_cols), oh_te, te_te, code_te], axis=1)
X_train = X_train.apply(pd.to_numeric, errors='coerce').fillna(0.0)
X_test  = X_test.apply(pd.to_numeric, errors='coerce').fillna(0.0)
X_train, X_test = X_train.align(X_test, join='inner', axis=1)
print('X_train:', X_train.shape, '  X_test:', X_test.shape)

## 4. Base models — LightGBM / XGBoost / CatBoost × (Task A, Task B)

5-fold StratifiedKFold on RiskTier, same splits reused for Task B so OOF arrays align across tasks.

In [ ]:
folds = list(skf.split(np.arange(len(X_train)), y_tier))

def fit_lgb_cls(X, y, Xt, folds, rounds=3000):
    p = dict(objective='multiclass', num_class=N_CLASSES, metric='multi_logloss',
             learning_rate=0.05, num_leaves=63, feature_fraction=0.85,
             bagging_fraction=0.85, bagging_freq=5, min_child_samples=20,
             lambda_l2=1.0, verbose=-1, seed=SEED)
    oof = np.zeros((len(X),N_CLASSES)); tp = np.zeros((len(Xt),N_CLASSES))
    for tri,vai in folds:
        m = lgb.train(p, lgb.Dataset(X.iloc[tri],y.iloc[tri]), rounds,
                      valid_sets=[lgb.Dataset(X.iloc[vai],y.iloc[vai])],
                      callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)])
        oof[vai] = m.predict(X.iloc[vai], num_iteration=m.best_iteration)
        tp += m.predict(Xt, num_iteration=m.best_iteration)/len(folds)
    return oof, tp

def fit_lgb_reg(X, y, Xt, folds, rounds=3000):
    p = dict(objective='regression', metric='rmse', learning_rate=0.05, num_leaves=63,
             feature_fraction=0.85, bagging_fraction=0.85, bagging_freq=5,
             min_child_samples=20, lambda_l2=1.0, verbose=-1, seed=SEED)
    oof = np.zeros(len(X)); tp = np.zeros(len(Xt))
    for tri,vai in folds:
        m = lgb.train(p, lgb.Dataset(X.iloc[tri],y.iloc[tri]), rounds,
                      valid_sets=[lgb.Dataset(X.iloc[vai],y.iloc[vai])],
                      callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)])
        oof[vai] = m.predict(X.iloc[vai], num_iteration=m.best_iteration)
        tp += m.predict(Xt, num_iteration=m.best_iteration)/len(folds)
    return oof, tp

def fit_xgb_cls(X, y, Xt, folds, rounds=3000):
    p = dict(objective='multi:softprob', num_class=N_CLASSES, eval_metric='mlogloss',
             learning_rate=0.05, max_depth=7, subsample=0.85, colsample_bytree=0.85,
             min_child_weight=5, reg_lambda=1.0, tree_method='hist',
             random_state=SEED, verbosity=0)
    oof = np.zeros((len(X),N_CLASSES)); tp = np.zeros((len(Xt),N_CLASSES))
    dt = xgb.DMatrix(Xt)
    for tri,vai in folds:
        d_tr, d_va = xgb.DMatrix(X.iloc[tri],label=y.iloc[tri]), xgb.DMatrix(X.iloc[vai],label=y.iloc[vai])
        m = xgb.train(p, d_tr, rounds, [(d_va,'v')], early_stopping_rounds=150, verbose_eval=False)
        oof[vai] = m.predict(d_va, iteration_range=(0, m.best_iteration+1))
        tp += m.predict(dt, iteration_range=(0, m.best_iteration+1))/len(folds)
    return oof, tp

def fit_xgb_reg(X, y, Xt, folds, rounds=3000):
    p = dict(objective='reg:squarederror', eval_metric='rmse', learning_rate=0.05,
             max_depth=7, subsample=0.85, colsample_bytree=0.85, min_child_weight=5,
             reg_lambda=1.0, tree_method='hist', random_state=SEED, verbosity=0)
    oof = np.zeros(len(X)); tp = np.zeros(len(Xt))
    dt = xgb.DMatrix(Xt)
    for tri,vai in folds:
        d_tr, d_va = xgb.DMatrix(X.iloc[tri],label=y.iloc[tri]), xgb.DMatrix(X.iloc[vai],label=y.iloc[vai])
        m = xgb.train(p, d_tr, rounds, [(d_va,'v')], early_stopping_rounds=150, verbose_eval=False)
        oof[vai] = m.predict(d_va, iteration_range=(0, m.best_iteration+1))
        tp += m.predict(dt, iteration_range=(0, m.best_iteration+1))/len(folds)
    return oof, tp

def fit_cat_cls(X, y, Xt, folds, iters=2000):
    oof = np.zeros((len(X),N_CLASSES)); tp = np.zeros((len(Xt),N_CLASSES))
    for tri,vai in folds:
        m = CatBoostClassifier(loss_function='MultiClass', classes_count=N_CLASSES,
                                iterations=iters, learning_rate=0.05, depth=7,
                                l2_leaf_reg=3.0, random_seed=SEED, verbose=False,
                                allow_writing_files=False, early_stopping_rounds=150)
        m.fit(X.iloc[tri],y.iloc[tri], eval_set=(X.iloc[vai],y.iloc[vai]), use_best_model=True)
        oof[vai] = m.predict_proba(X.iloc[vai])
        tp += m.predict_proba(Xt)/len(folds)
    return oof, tp

def fit_cat_reg(X, y, Xt, folds, iters=2000):
    oof = np.zeros(len(X)); tp = np.zeros(len(Xt))
    for tri,vai in folds:
        m = CatBoostRegressor(loss_function='RMSE', iterations=iters, learning_rate=0.05,
                               depth=7, l2_leaf_reg=3.0, random_seed=SEED, verbose=False,
                               allow_writing_files=False, early_stopping_rounds=150)
        m.fit(X.iloc[tri],y.iloc[tri], eval_set=(X.iloc[vai],y.iloc[vai]), use_best_model=True)
        oof[vai] = m.predict(X.iloc[vai])
        tp += m.predict(Xt)/len(folds)
    return oof, tp

In [ ]:
# Task A — classification
oof_A, test_A = {}, {}
oof_A['lgb'],     test_A['lgb']     = fit_lgb_cls(X_train, y_tier, X_test, folds)
oof_A['xgb'],     test_A['xgb']     = fit_xgb_cls(X_train, y_tier, X_test, folds)
oof_A['cat'],     test_A['cat']     = fit_cat_cls(X_train, y_tier, X_test, folds)
# Ordinal trick: regression on tier → round/clip → one-hot for ensemble compatibility.
ord_oof, ord_test = fit_lgb_reg(X_train, y_tier.astype(float), X_test, folds)
ord_oof_cls  = np.clip(np.round(ord_oof).astype(int), 0, N_CLASSES-1)
ord_test_cls = np.clip(np.round(ord_test).astype(int), 0, N_CLASSES-1)
oof_A['lgb_ord']  = np.eye(N_CLASSES)[ord_oof_cls]
test_A['lgb_ord'] = np.eye(N_CLASSES)[ord_test_cls]

for k,v in oof_A.items():
    print(f'  {k:8s}  acc={accuracy_score(y_tier, v.argmax(1)):.4f}')

In [ ]:
# Task B — regression
oof_B, test_B = {}, {}
oof_B['lgb'], test_B['lgb'] = fit_lgb_reg(X_train, y_rate, X_test, folds)
oof_B['xgb'], test_B['xgb'] = fit_xgb_reg(X_train, y_rate, X_test, folds)
oof_B['cat'], test_B['cat'] = fit_cat_reg(X_train, y_rate, X_test, folds)
for k,v in oof_B.items():
    print(f'  {k:8s}  R²={r2_score(y_rate, v):.4f}')

## 5. Stage-2 cross-task stacking + ensemble weights

In [ ]:
def build_aug(X_train, X_test, oof_A, test_A, oof_B, test_B, ord_oof, ord_test):
    parts_tr = [X_train.reset_index(drop=True)]; parts_te = [X_test.reset_index(drop=True)]
    for m,a in oof_A.items():
        cols = [f'oofA_{m}_p{k}' for k in range(a.shape[1])]
        parts_tr.append(pd.DataFrame(a, columns=cols)); parts_te.append(pd.DataFrame(test_A[m], columns=cols))
    parts_tr.append(pd.DataFrame({'oofA_lgb_ord_float': ord_oof}))
    parts_te.append(pd.DataFrame({'oofA_lgb_ord_float': ord_test}))
    for m,a in oof_B.items():
        parts_tr.append(pd.DataFrame({f'oofB_{m}': a}))
        parts_te.append(pd.DataFrame({f'oofB_{m}': test_B[m]}))
    return pd.concat(parts_tr, axis=1), pd.concat(parts_te, axis=1)

X_aug_tr, X_aug_te = build_aug(X_train, X_test, oof_A, test_A, oof_B, test_B, ord_oof, ord_test)
print('X_aug_tr:', X_aug_tr.shape)

# Stage-2 LGBM per task
def stage2(Xa, y, Xat, folds, task):
    p = dict(learning_rate=0.03, num_leaves=31, min_child_samples=30,
             feature_fraction=0.8, bagging_fraction=0.85, bagging_freq=5,
             lambda_l2=1.0, verbose=-1, seed=SEED)
    p.update(objective='multiclass', num_class=N_CLASSES, metric='multi_logloss') if task=='A' \
        else p.update(objective='regression', metric='rmse')
    oof = np.zeros((len(Xa), N_CLASSES)) if task=='A' else np.zeros(len(Xa))
    tp  = np.zeros((len(Xat), N_CLASSES)) if task=='A' else np.zeros(len(Xat))
    for tri,vai in folds:
        m = lgb.train(p, lgb.Dataset(Xa.iloc[tri],y.iloc[tri]), 2000,
                      valid_sets=[lgb.Dataset(Xa.iloc[vai],y.iloc[vai])],
                      callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)])
        oof[vai] = m.predict(Xa.iloc[vai], num_iteration=m.best_iteration)
        tp += m.predict(Xat, num_iteration=m.best_iteration)/len(folds)
    return oof, tp

s2_A_oof, s2_A_test = stage2(X_aug_tr, y_tier, X_aug_te, folds, 'A')
s2_B_oof, s2_B_test = stage2(X_aug_tr, y_rate, X_aug_te, folds, 'B')
print(f'Stage-2 A acc={accuracy_score(y_tier, s2_A_oof.argmax(1)):.4f}   R²={r2_score(y_rate, s2_B_oof):.4f}')

oof_A['stack2']  = s2_A_oof;  test_A['stack2']  = s2_A_test
oof_B['stack2']  = s2_B_oof;  test_B['stack2']  = s2_B_test

In [ ]:
# --- Optimise convex weights on OOF ---
def opt_blend_A(oofs, y):
    names = list(oofs.keys()); probs = np.stack([oofs[n] for n in names])
    best = (-1, None); step = 0.1; grid = np.arange(0, 1+step/2, step)
    if len(names) == 5:
        for a in grid:
            for b in grid:
                for c in grid:
                    for d in grid:
                        e = 1-a-b-c-d
                        if e < -1e-9: continue
                        e = max(e, 0)
                        w = np.array([a,b,c,d,e]); s = w.sum()
                        if s == 0: continue
                        w = w/s
                        acc = accuracy_score(y, (w[:,None,None]*probs).sum(0).argmax(1))
                        if acc > best[0]: best = (acc, w)
    else:
        best = (accuracy_score(y, np.mean(probs, axis=0).argmax(1)), np.ones(len(names))/len(names))
    return dict(zip(names, best[1])), best[0]

def opt_blend_B(oofs, y):
    names = list(oofs.keys()); P = np.column_stack([oofs[n] for n in names])
    def neg_r2(w):
        w = np.clip(w,0,None); s = w.sum()
        if s == 0: return 0.0
        return -r2_score(y, P @ (w/s))
    res = minimize(neg_r2, np.ones(len(names))/len(names), method='Nelder-Mead',
                   options=dict(maxiter=800, xatol=1e-4, fatol=1e-5))
    w = np.clip(res.x,0,None); w = w/w.sum()
    return dict(zip(names, w)), -neg_r2(w)

w_A, acc_final = opt_blend_A(oof_A, y_tier.to_numpy())
w_B, r2_final  = opt_blend_B(oof_B, y_rate.to_numpy())
combined = 0.5*acc_final + 0.5*r2_final
print(f'Final OOF  acc={acc_final:.4f}  R²={r2_final:.4f}  combined={combined:.4f}')
print('Weights A:', {k: round(v,3) for k,v in w_A.items()})
print('Weights B:', {k: round(v,3) for k,v in w_B.items()})

## 6. Submission

In [ ]:
final_test_A = sum(w_A[n]*test_A[n] for n in w_A)
final_test_B = sum(w_B[n]*test_B[n] for n in w_B)

tier_pred = final_test_A.argmax(1).astype(int)
rate_pred = np.round(np.clip(final_test_B, RATE_MIN, RATE_MAX), 2)

sub = pd.DataFrame({'Id': ids_test.astype(int).values,
                    'RiskTier': tier_pred, 'InterestRate': rate_pred})
assert sub['Id'].is_unique and len(sub) == 15000
assert set(sub['RiskTier']).issubset(set(range(5)))
assert sub['InterestRate'].between(RATE_MIN, RATE_MAX).all()
sub.to_csv('submission.csv', index=False)
print('submission.csv written; head:')
print(sub.head())
print('tier distribution:', sub['RiskTier'].value_counts().to_dict())